<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 04 support · Expected correction values</h1></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>مساند اللاب 04 · قيم التصحيح المتوقعة</h1></td></tr></tbody>
</table>

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><a href="README.md">Day 3</a> · <a href="labs/lab04/WALKTHROUGH.md">Walkthrough</a></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><a href="README.md">اليوم الثالث</a> · <a href="labs/lab04/WALKTHROUGH.md">الشرح التطبيقي</a></td></tr></tbody>
</table>



<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and scope</h2><p>Calculate the expected business change directly from the fixed data. This notebook runs Python only: it does not create Delta versions, execute ACID transactions or simulate a successful engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والنطاق</h2><p>احسب التغيير المتوقع مباشرة من البيانات الثابتة. يشغل الدفتر Python فقط؛ لا ينشئ نسخ Delta، ولا ينفذ معاملات ACID، ولا يحاكي محركًا ناجحًا.</p></td></tr></tbody>
</table>



<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Locate the source</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. تحديد المصدر</h2></td></tr></tbody>
</table>



In [1]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset, new_workspace, write_json
require_fixed_dataset(SOURCE)
from masar.delta_reference import day03_reference, revision_merge
from masar.silver_reference import reference_result
WORK = new_workspace(ROOT, "day03_reference")
print("Source checked | MASAR_SMALL_V1 | Reference business values only")

Source checked | MASAR_SMALL_V1 | Reference business values only


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Compare one changed trip</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. مقارنة الرحلة المعدلة</h2></td></tr></tbody>
</table>



In [2]:
report = day03_reference(SOURCE)
for label in ("corrected_trip_before", "corrected_trip_after"):
    row = report[label]
    print(label, {key: row[key] for key in ("trip_id", "fare_sar", "source_revision")})
assert report["before"]["rows"] == report["after"]["rows"] == 75

corrected_trip_before {'trip_id': 'SYN_T0001', 'fare_sar': '18.00', 'source_revision': 1}
corrected_trip_after {'trip_id': 'SYN_T0001', 'fare_sar': '23.00', 'source_revision': 2}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Reconcile the totals</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. تسوية المجاميع</h2></td></tr></tbody>
</table>



In [3]:
from decimal import Decimal
change = Decimal(report["after"]["fare_sar"]) - Decimal(report["before"]["fare_sar"])
assert change == Decimal("5.00")
print("Expected total before:", report["before"]["fare_sar"], "SAR")
print("Expected total after: ", report["after"]["fare_sar"], "SAR")
print("Expected change:      ", change, "SAR")

Expected total before: 1875.60 SAR
Expected total after:  1880.60 SAR
Expected change:       5.00 SAR


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Check replay and older data</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>4. فحص الإعادة والبيانات الأقدم</h2></td></tr></tbody>
</table>



In [4]:
assert report["after"]["digest"] == report["replay_digest"] == report["stale_replay_digest"]
print("Identical correction replay preserves values: True")
print("Older revision did not undo the correction: True")
print("Older trip revision ignored:", report["stale_ignored_count"])

Identical correction replay preserves values: True
Older revision did not undo the correction: True
Older trip revision ignored: 1


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>5. Check the schema scenario meaning</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>5. فحص معنى سيناريو المخطط</h2></td></tr></tbody>
</table>



In [5]:
fixture = report["schema_fixture"]
print(json.dumps({k:fixture[k] for k in ("trip_id","extra_column","spark_type","value","meaning")}, indent=2))
print("Expected sandbox layout: 74 existing null surcharges + 1 supplied value")
print("No schema change was executed in this reference notebook.")

{
  "trip_id": "SYN_T0002",
  "extra_column": "surcharge_sar",
  "spark_type": "decimal(12,2)",
  "value": "2.00",
  "meaning": "Separate surcharge; do not silently redefine fare_sar."
}
Expected sandbox layout: 74 existing null surcharges + 1 supplied value
No schema change was executed in this reference notebook.


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>6. Save verified arithmetic, not engine claims</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>6. حفظ الحساب المتحقق لا ادعاءات المحرك</h2></td></tr></tbody>
</table>



In [6]:
assert all(v is True for v in report["checks"].values())
assert report["engine_executed"] is False
write_json(WORK / "day03_reference.json", report)
print("Reference checks passed:", len(report["checks"]))
print("Engine executed:", report["engine_executed"])
print("Next: the actual Lab 4a and 4b notebooks, once the runtime is provisioned.")

Reference checks passed: 15
Engine executed: False
Next: the actual Lab 4a and 4b notebooks, once the runtime is provisioned.
